In [ ]:
# 🔹 1. Import Libraries
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt


In [ ]:
# 🔹 2. Load Dataset (Sensor Data)
df = pd.read_csv("/content/railways detection.csv")
print(df.head())


   class index                                          filepaths     labels  \
0            0  train/Defective/131004959_171473721383413_8222...  Defective   
1            0  train/Defective/131051004_382613492830631_1306...  Defective   
2            0  train/Defective/131065180_999185033824671_7735...  Defective   
3            0  train/Defective/131084537_190751489449739_2798...  Defective   
4            0  train/Defective/131092897_149705809860659_8798...  Defective   

  data set  
0    train  
1    train  
2    train  
3    train  
4    train  


In [ ]:
# 🔹 3. Define Input Shapes
IMG_SHAPE = (224, 224, 3)   # Image input
SENSOR_DIM = 5              # Example sensor features

In [ ]:
# 🔹 4. Preprocessing Function
def preprocess_inputs(image_batch, sensor_batch):
    # Normalize images
    image_batch = image_batch.astype('float32') / 255.0

    # Standardize sensor data
    sensor_batch = (sensor_batch - np.mean(sensor_batch)) / np.std(sensor_batch)

    return image_batch, sensor_batch

print("✅ Data preprocessing completed")


✅ Data preprocessing completed


In [ ]:
# =========================================
# 🔹 5. Build Multi-Modal Model
# =========================================

def build_model():

    # 🔸 CNN Branch (Image Input)
    image_input = layers.Input(shape=IMG_SHAPE, name='image_input')

    x = layers.Conv2D(32, (3,3), activation='relu')(image_input)
    x = layers.MaxPooling2D(2,2)(x)

    x = layers.Conv2D(64, (3,3), activation='relu')(x)
    x = layers.GlobalAveragePooling2D()(x)

    # 🔸 MLP Branch (Sensor Input)
    sensor_input = layers.Input(shape=(SENSOR_DIM,), name='sensor_input')

    y = layers.Dense(16, activation='relu')(sensor_input)
    y = layers.Dense(8, activation='relu')(y)

    # 🔸 Fusion Layer
    combined = layers.concatenate([x, y])

    # 🔸 Final Layers
    z = layers.Dense(64, activation='relu')(combined)
    z = layers.Dropout(0.3)(z)

    output = layers.Dense(1, activation='sigmoid')(z)

    # 🔸 Model
    model = models.Model(inputs=[image_input, sensor_input], outputs=output)

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

# Build model
model = build_model()
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image_input         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 222, 222,  │        896 │ image_input[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 111, 111,  │          0 │ conv2d[0][0]      │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sensor_input        │ (None, 5)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 109, 109,  │     18,496 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 16)        │         96 │ sensor_input[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 64)        │          0 │ conv2d_1[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 8)         │        136 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 72)        │          0 │ global_average_p… │
│ (Concatenate)       │                   │            │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 64)        │      4,672 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 64)        │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 1)         │         65 │ dropout[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 24,361 (95.16 KB)

 Trainable params: 24,361 (95.16 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# 🔹 6. Dummy Training (Simulation)
# =========================================

# Simulated data (replace with real dataset)
X_images = np.random.rand(100, 224, 224, 3)
X_sensor = np.random.rand(100, SENSOR_DIM)
y = np.random.randint(0, 2, 100)

In [ ]:
# Train model
history = model.fit(
    [X_images, X_sensor],
    y,
    epochs=5,
    batch_size=16
)


Epoch 1/5
7/7 ━━━━━━━━━━━━━━━━━━━━ 11s 998ms/step - accuracy: 0.5300 - loss: 0.6978
Epoch 2/5
7/7 ━━━━━━━━━━━━━━━━━━━━ 9s 1s/step - accuracy: 0.5600 - loss: 0.6913
Epoch 3/5
7/7 ━━━━━━━━━━━━━━━━━━━━ 7s 968ms/step - accuracy: 0.5300 - loss: 0.6899
Epoch 4/5
7/7 ━━━━━━━━━━━━━━━━━━━━ 10s 971ms/step - accuracy: 0.5200 - loss: 0.6939
Epoch 5/5
7/7 ━━━━━━━━━━━━━━━━━━━━ 11s 934ms/step - accuracy: 0.5400 - loss: 0.6844


In [ ]:
# =========================================
# 🔹 7. Prediction Function
# =========================================

def monitor_tracks(model, img, sensor_data):
    prediction = model.predict([img, sensor_data])
    confidence = prediction[0][0]

    if confidence > 0.5:
        print(f"🚨 ALERT: Fault Detected! Confidence: {confidence:.2f}")
    else:
        print(f"✅ Track Safe. Confidence: {confidence:.2f}")

# Test prediction
test_img = np.random.rand(1, 224, 224, 3)
test_sensor = np.random.rand(1, SENSOR_DIM)

monitor_tracks(model, test_img, test_sensor)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 143ms/step
🚨 ALERT: Fault Detected! Confidence: 0.55
